# 22 Check Audio Heads Trainer

In [1]:
import time
import pandas as pd

from wooftrainer.trainers.audio_heads import (
    balanced_weights,
    choose_C,
    clip_weights,
    fit_temperature,
    make_folds,
    out_of_fold_logits,
    score_probs
)
from wooftrainer.trainers.embeddings import attach_labels, load_embeddings
from woofdecoder.branches.audio import select_layer, softmax
from wooftrainer.trainers.audio_heads import train_audio_heads

## Data and Model Preparation

In [2]:
x, table = load_embeddings("audio_emotion", "train", "beats_iter3_plus_as2m")
table = attach_labels(table, "audio_emotion", "train")
features = select_layer(x, "mean")

labels = table["arousal"].to_numpy()
weights = balanced_weights(labels, clip_weights(table))
folds = make_folds(labels)

## Optimizing C_arousal Value

In [3]:
best_C, c_table = choose_C(features, labels, weights, folds)
print(c_table.round(3).to_string(index=False), "\nbest C:", best_C)

 log_loss  macro_f1     C
    1.038     0.543 0.000
    0.976     0.562 0.000
    0.922     0.564 0.000
    0.890     0.572 0.001
    0.899     0.575 0.003
    0.968     0.549 0.010
    1.438     0.521 0.100
    2.415     0.515 1.000 
best C: 0.001


- at C = 1.0, the probe default settings show that the prediction is badly over fitting because log loss is super high (2.415)
- at C = 0.001, the scores show the best of both f1 (0.515 --> 0.572) and log_loss (2.415 --> 0.890).

Thus, we can trust the aurosal probabilities and the strong regularisation stopped the memorizing.

## Optimizing T_arousal Value

In [4]:
logits, classes = out_of_fold_logits(features, labels, weights, folds, best_C)
temperature = fit_temperature(logits, labels, classes, weights)
print("temperature:", round(temperature, 2))
print("before:", score_probs(labels, softmax(logits), classes, weights))
print("after:", score_probs(labels, softmax(logits/temperature), classes, weights))

temperature: 0.86
before: {'log_loss': 0.8902433376411953, 'macro_f1': 0.5715719412551769}
after: {'log_loss': 0.8871376437531784, 'macro_f1': 0.5715719412551769}


## Full Audio Train Check

In [7]:
start = time.perf_counter()
heads = train_audio_heads("beats_iter3_plus_as2m", "mean")
print(f"took {time.perf_counter() - start:.0f}s\n")
print(heads.summary.round(3).to_string(index=False), "\n")
print(heads.tables["C valence"].round(3).to_string(index=False), "\n")
print(heads.tables["C situation"].round(3).to_string(index=False), "\n")
print(heads.tables["nudge"].round(3).to_string(index=False), "\n")
print(pd.DataFrame(heads.nudge.weights, index=heads.nudge.classes, columns=heads.nudge.inputs).round(2).to_string())

took 104s

     head     C    l2  temperature  before_log_loss  before_macro_f1  after_log_loss  after_macro_f1
  arousal 0.001              0.857            0.890            0.572           0.887           0.572
  valence 0.000              0.831            1.038            0.461           1.036           0.461
situation 0.000  0.01        0.842            1.154            0.519           1.140           0.509 

 log_loss  macro_f1     C
    1.070     0.446 0.000
    1.051     0.453 0.000
    1.038     0.461 0.000
    1.039     0.468 0.001
    1.072     0.460 0.003
    1.180     0.444 0.010
    1.856     0.435 0.100
    3.329     0.426 1.000 

 log_loss  macro_f1     C
    1.234     0.463 0.000
    1.176     0.499 0.000
    1.154     0.519 0.000
    1.160     0.529 0.001
    1.194     0.530 0.003
    1.269     0.514 0.010
    1.483     0.471 0.100
    1.667     0.419 1.000 

 log_loss  macro_f1    l2
    1.154     0.519  none
    1.146     0.509  0.01
    1.150     0.516   0.1
    1.1

- Arousal, from optimizing C_arousal value and T_arousal value, we can see that best C is equal to 0.001, and T is around 0.86